In [2]:
import pandas as pd
import duckdb
data = [
    # Device A
    ["A", "2026-07-01 08:00:00", "NORMAL"],
    ["A", "2026-07-01 08:01:00", "ERROR"],
    ["A", "2026-07-01 08:02:00", "ERROR"],
    ["A", "2026-07-01 08:03:00", "NORMAL"],
    ["A", "2026-07-01 08:04:00", "ERROR"],
    ["A", "2026-07-01 08:05:00", "ERROR"],
    ["A", "2026-07-01 08:06:00", "ERROR"],

    # Device B
    ["B", "2026-07-01 08:00:00", "NORMAL"],
    ["B", "2026-07-01 08:01:00", "ERROR"],
    ["B", "2026-07-01 08:02:00", "NORMAL"],
    ["B", "2026-07-01 08:03:00", "ERROR"],
    ["B", "2026-07-01 08:04:00", "ERROR"],
    ["B", "2026-07-01 08:05:00", "NORMAL"],

    # Device C
    ["C", "2026-07-01 08:00:00", "ERROR"],
    ["C", "2026-07-01 08:01:00", "ERROR"],
    ["C", "2026-07-01 08:02:00", "ERROR"],
    ["C", "2026-07-01 08:03:00", "NORMAL"],
]

df = pd.DataFrame(
    data,
    columns=["device_id", "collect_time", "status"]
)

df["collect_time"] = pd.to_datetime(df["collect_time"])

print(df)



   device_id        collect_time  status
0          A 2026-07-01 08:00:00  NORMAL
1          A 2026-07-01 08:01:00   ERROR
2          A 2026-07-01 08:02:00   ERROR
3          A 2026-07-01 08:03:00  NORMAL
4          A 2026-07-01 08:04:00   ERROR
5          A 2026-07-01 08:05:00   ERROR
6          A 2026-07-01 08:06:00   ERROR
7          B 2026-07-01 08:00:00  NORMAL
8          B 2026-07-01 08:01:00   ERROR
9          B 2026-07-01 08:02:00  NORMAL
10         B 2026-07-01 08:03:00   ERROR
11         B 2026-07-01 08:04:00   ERROR
12         B 2026-07-01 08:05:00  NORMAL
13         C 2026-07-01 08:00:00   ERROR
14         C 2026-07-01 08:01:00   ERROR
15         C 2026-07-01 08:02:00   ERROR
16         C 2026-07-01 08:03:00  NORMAL


## 题目要求

* **分别使用 SQL 和 Pandas 完成。**

## 需求

* **找出每个设备中：**

- 连续 ERROR 次数达到 2 次及以上的所有时间段。

- 输出结果：

|device_id|error_start_time|error_end_time|error_duration_count|
|---------|----------------|--------------|--------------------|
|A	|2026-07-01 08:01:00|	2026-07-01 08:02:00|	2|
|A	|2026-07-01 08:04:00|	2026-07-01 08:06:00|	3|
|B	|2026-07-01 08:03:00|   2026-07-01 08:04:00|	2|
|C	|2026-07-01 08:00:00|	2026-07-01 08:02:00|	3|

In [45]:
# SQL轨道

query =  """

WITH is_error_table AS (
SELECT 
    device_id,
    collect_time,
    status,
    CASE WHEN status = 'ERROR' THEN True ELSE False END AS is_error
FROM df


),
last_is_error AS(
SELECT
    device_id,
    collect_time,
    status,
    is_error,
    COALESCE(LAG(is_error,1) OVER(PARTITION BY device_id ORDER BY collect_time) , False)AS preview_is_error
FROM is_error_table
),
error_start_sign AS (
SELECT
    device_id,
    collect_time,
    status,
    is_error,
    preview_is_error,
    CASE WHEN is_error = True AND preview_is_error = False THEN 1 ELSE 0 END  AS error_start
FROM last_is_error
),
phase_sign_table AS (
SELECT
    device_id,
    collect_time,
    status,
    SUM(error_start) OVER(PARTITION BY device_id ORDER BY collect_time)::INTEGER AS phase_sign
FROM error_start_sign
WHERE STATUS = 'ERROR'
),

consecutive_error AS(
SELECT
    device_id,
    MIN(collect_time) AS error_start_time,
    MAX(collect_time) AS error_end_time,
    COUNT(*) AS error_duration_count
FROM phase_sign_table
GROUP BY device_id,phase_sign
)

SELECT *
FROM consecutive_error
WHERE error_duration_count >= 2
ORDER BY device_id
"""

df_sql = duckdb.execute(query).fetchdf()
df_sql

,device_id,error_start_time,error_end_time,error_duration_count
0,A,2026-07-01 08:01:00,2026-07-01 08:02:00,2
1,A,2026-07-01 08:04:00,2026-07-01 08:06:00,3
2,B,2026-07-01 08:03:00,2026-07-01 08:04:00,2
3,C,2026-07-01 08:00:00,2026-07-01 08:02:00,3


In [44]:
# # pandas轨道
df = df.sort_values(by=['device_id','collect_time'])
df['is_error'] = df['status'] == 'ERROR'
df['preview_error'] = df.groupby('device_id')['is_error'].shift(1).fillna(False)
df['error_start'] = ((df['is_error']) & (df['preview_error'] == False)).astype(int)
df['phase_sign'] = df.groupby('device_id')['error_start'].cumsum()

df_pd = (
    df
    .loc[df['is_error']]
    .groupby(['device_id','phase_sign'])
    .agg(
        error_start_time = ('collect_time','min'),
        error_end_time = ('collect_time','max'),
        error_duration_count = ('device_id','size')
    )
    .query("error_duration_count >= 2")
    .reset_index()
    [['device_id','error_start_time','error_end_time','error_duration_count']]
)

df_pd

,device_id,error_start_time,error_end_time,error_duration_count
0,A,2026-07-01 08:01:00,2026-07-01 08:02:00,2
1,A,2026-07-01 08:04:00,2026-07-01 08:06:00,3
2,B,2026-07-01 08:03:00,2026-07-01 08:04:00,2
3,C,2026-07-01 08:00:00,2026-07-01 08:02:00,3
